In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [3]:
!pip install -q mlflow dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 71.9 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 73.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [16]:

import pandas as pd
import mlflow
import dagshub

dagshub.init(repo_owner='ZukaCS', repo_name='ML_assignment_2', mlflow=True)

PATH = "/kaggle/input/competitions/ieee-fraud-detection/"

print("Loading XGBoost model from Model Registry...")
model = mlflow.sklearn.load_model("models:/IEEE_Fraud_XGBoost/1")
print("Model loaded.")

print("\nLoading Kaggle test data...")
test_tx = pd.read_csv(PATH + "test_transaction.csv")
test_id = pd.read_csv(PATH + "test_identity.csv")
test_id.columns = test_id.columns.str.replace('-', '_')  
test_raw = test_tx.merge(test_id, on="TransactionID", how="left")
print(f"Test data shape: {test_raw.shape}")


print("\nGenerating predictions...")
test_proba = model.predict_proba(test_raw)[:, 1]
print(f"Predicted {len(test_proba):,} rows.")
print(f"Probability stats: min={test_proba.min():.4f}, max={test_proba.max():.4f}, mean={test_proba.mean():.4f}")


submission = pd.DataFrame({
    'TransactionID': test_raw['TransactionID'],
    'isFraud':       test_proba,
})
submission.to_csv('submission.csv', index=False)

print(f"\nSubmission saved to submission.csv")
print(f"Submission shape: {submission.shape}")
print(submission.head())

Initialized MLflow to track repo "ZukaCS/ML_assignment_2"

Repository ZukaCS/ML_assignment_2 initialized!

Loading XGBoost model from Model Registry...


Model loaded.

Loading Kaggle test data...
Test data shape: (506691, 433)

Generating predictions...
Predicted 506,691 rows.
Probability stats: min=0.0002, max=0.9996, mean=0.1738

Submission saved to submission.csv
Submission shape: (506691, 2)
   TransactionID   isFraud
0        3663549  0.043641
1        3663550  0.022948
2        3663551  0.068993
3        3663552  0.048639
4        3663553  0.024894
